In [1]:
import time

# Start timer
start_time = time.time()

In [2]:
import pandas as pd
import numpy as np
import json
import requests
import os
from dotenv import load_dotenv

In [3]:
# Load environment variables
load_dotenv()
api_key = os.getenv("GROQ_API_KEY")

In [4]:
# Load the data
n = 100
df = pd.read_parquet('data/bank_reviews_processed.parquet').head(n)
df = df[df['detected_language'] == 'English'].reset_index(drop=True)

In [5]:
# Define system prompt for classification
ENHANCED_SYSTEM_PROMPT = """Analyze the following bank review written in English, Tagalog, or Taglish. Some texts include shortened terms such as 'aq' for 'ako', 'pd' for 'pwede', 'pw' for password, 'w8' for 'wait', and similar context.

Your task has TWO parts:

PART 1 - TOPIC IDENTIFICATION: 
Analyze the reviews and classify all topics which must only consist of the following: 'Accessibility', 'Security', 'Convenience', and 'Customer Support'. 
Consider all aspects including implicit mentions and context.

PART 2 - SATISFACTION RATING:
For each identified topic, rate the customer's satisfaction level from 1-5:
- 1: Very Dissatisfied (very negative experience)
- 2: Dissatisfied (negative experience) 
- 3: Neutral (mixed or unclear sentiment)
- 4: Satisfied (positive experience)
- 5: Very Satisfied (very positive experience)

RESPONSE FORMAT (CRITICAL - Follow exactly):
Return your response in this exact JSON format:
{
  "topics": ["topic1", "topic2"],
  "ratings": {"topic1": rating_number, "topic2": rating_number}
}

IMPORTANT: Only include the classifications mentioned from reviews with implied or explicit contexts. STRICTLY return the exact valid JSON only, NO OTHER TEXT."""

print("🚀 ENHANCED ZERO-SHOT CLASSIFICATION + RATING SYSTEM")
print("=" * 60)
print("✅ Enhanced system prompt created for classification + rating")
print("📋 This enhanced workflow will:")
print("   1. Identify relevant topics (Accessibility, Security, Convenience, Customer Support)")
print("   2. Rate satisfaction 1-5 for each identified topic")
print("   3. Return structured JSON response")
print("   4. Create rating columns for each topic (1-5 scale)")
print("   5. Create missing indicator columns (0=present, 1=missing)")
print("   6. Generate comprehensive analysis and exports")
print("=" * 60)
print("🔄 OVERWRITING OLD FUNCTIONALITY - Using enhanced features only")
print("✓ Setup completed")
print(f"✓ DataFrame loaded: {df.shape}")
print(f"✓ API key configured: {'Yes' if api_key else 'No'}")

🚀 ENHANCED ZERO-SHOT CLASSIFICATION + RATING SYSTEM
✅ Enhanced system prompt created for classification + rating
📋 This enhanced workflow will:
   1. Identify relevant topics (Accessibility, Security, Convenience, Customer Support)
   2. Rate satisfaction 1-5 for each identified topic
   3. Return structured JSON response
   4. Create rating columns for each topic (1-5 scale)
   5. Create missing indicator columns (0=present, 1=missing)
   6. Generate comprehensive analysis and exports
🔄 OVERWRITING OLD FUNCTIONALITY - Using enhanced features only
✓ Setup completed
✓ DataFrame loaded: (100, 20)
✓ API key configured: Yes


In [6]:
import hashlib
import tiktoken

def create_batch_json(df, system_prompt, batch_size=None):
    output_file = 'data/output_proceeds/zeroshot_batch.jsonl'
    
    # Ensure the output directory exists
    output_dir = os.path.dirname(output_file)
    os.makedirs(output_dir, exist_ok=True)
    
    # Pick batch size
    if batch_size is None:
        df_sample = df.copy()
        print(f"📝 Processing ALL {len(df):,} reviews")
    else:
        df_sample = df.head(batch_size).copy()
        print(f"📝 Processing {len(df_sample):,} reviews (limited from {len(df):,} total)")
    
    jsonl_content = []
    mapping = {}

    # Build the shared prefix once
    shared_prefix = [
        {"role": "system", "content": system_prompt.strip()}
    ]

    # Compute a hash of the cacheable prefix (so you can confirm it’s identical everywhere)
    prefix_str = json.dumps(shared_prefix, ensure_ascii=False, separators=(",", ":"))
    prefix_hash = hashlib.md5(prefix_str.encode("utf-8")).hexdigest()
    print(f"🔑 Cacheable prefix hash: {prefix_hash}")

    # --- Tokenizer for estimates ---
    enc = tiktoken.encoding_for_model("gpt-3.5-turbo")
    total_system_tokens = len(enc.encode(system_prompt))
    total_user_tokens = 0
    total_requests = 0

    for idx, row in df_sample.iterrows():
        custom_id = f"request-{idx}"
        user_content = str(row['processed_review']).strip()

        # Count tokens
        user_tokens = len(enc.encode(user_content))
        total_user_tokens += user_tokens
        total_requests += 1

        request_data = {
            "custom_id": custom_id,
            "method": "POST",
            "url": "/v1/chat/completions",
            "body": {
                "model": "openai/gpt-oss-20b",
                "messages": shared_prefix + [
                    {"role": "user", "content": user_content}
                ],
                "max_tokens": 512,
                "temperature": 0.0,
            }
        }

        jsonl_content.append(request_data)
        mapping[custom_id] = idx

    # Write JSONL
    with open(output_file, 'w', encoding='utf-8') as f:
        for item in jsonl_content:
            f.write(json.dumps(item, ensure_ascii=False) + '\n')

    # --- Stats summary ---
    total_prompt_tokens_without_cache = (total_system_tokens + (total_user_tokens // total_requests)) * total_requests
    total_prompt_tokens_with_cache = total_system_tokens + total_user_tokens

    print(f"✓ Batch JSON file created: {output_file} with {len(jsonl_content)} requests")
    print(f"✓ File size: {os.path.getsize(output_file)} bytes")

    print("\n--- TOKEN ESTIMATE ---")
    print(f"   System prompt tokens (once): {total_system_tokens}")
    print(f"   Avg user tokens per request: {total_user_tokens // total_requests}")
    print(f"   Total user tokens (all rows): {total_user_tokens}")
    print(f"   Total requests: {total_requests}")
    print(f"\n   If no caching: {total_prompt_tokens_without_cache:,} input tokens")
    print(f"   With caching:  {total_prompt_tokens_with_cache:,} input tokens")
    print(f"   => Estimated savings: {total_prompt_tokens_without_cache - total_prompt_tokens_with_cache:,} tokens")

    return output_file, mapping

try:
    # Create batch with the enhanced system prompt for classification + rating
    # Use the ENHANCED_SYSTEM_PROMPT instead of SYSTEM_PROMPT
    batch_file, id_mapping = create_batch_json(df, ENHANCED_SYSTEM_PROMPT, batch_size=len(df))
except Exception as e:
    print(f"Error creating batch JSON: {e}")
    batch_file, id_mapping = None, None

📝 Processing 100 reviews (limited from 100 total)
🔑 Cacheable prefix hash: 4c29df15e5925c454ef291ce623f37dd
✓ Batch JSON file created: data/output_proceeds/zeroshot_batch.jsonl with 100 requests
✓ File size: 167125 bytes

--- TOKEN ESTIMATE ---
   System prompt tokens (once): 277
   Avg user tokens per request: 51
   Total user tokens (all rows): 5169
   Total requests: 100

   If no caching: 32,800 input tokens
   With caching:  5,446 input tokens
   => Estimated savings: 27,354 tokens


In [7]:
# Check current dataset size and configure batch size
print("📊 Dataset Information:")
print(f"   Total Tagalog reviews: {len(df):,}")
print(f"   Current batch will process: {len(df):,} reviews")

# You can control batch size in several ways:

# Option 1: Process ALL reviews (current approach)
batch_size = len(df)  # Process everything
print(f"\n🔧 Batch Configuration:")
print(f"   Selected batch size: {batch_size:,} reviews")

# Option 2: Set a specific batch size limit (uncomment to use)
# max_batch_size = 5000  # Set your desired limit
# batch_size = min(len(df), max_batch_size)
# print(f"   Selected batch size: {batch_size:,} reviews (limited to {max_batch_size:,})")

# Option 3: Process in percentage of total data
# batch_percentage = 0.50  # Process 50% of data
# batch_size = int(len(df) * batch_percentage)
# print(f"   Selected batch size: {batch_size:,} reviews ({batch_percentage*100}% of total)")

print(f"\n💡 Note: Groq batch API can handle large batches efficiently.")
print(f"   Estimated processing time: {batch_size/100:.1f}-{batch_size/50:.1f} minutes")
print(f"   Cost estimate: ~${batch_size * 0.0001:.2f} (rough estimate)")

📊 Dataset Information:
   Total Tagalog reviews: 100
   Current batch will process: 100 reviews

🔧 Batch Configuration:
   Selected batch size: 100 reviews

💡 Note: Groq batch API can handle large batches efficiently.
   Estimated processing time: 1.0-2.0 minutes
   Cost estimate: ~$0.01 (rough estimate)


In [8]:
# ====================================================================
# STEP 3: BATCH PROCESSING WITH GROQ API
# ====================================================================

class GroqBatchProcessor:
    """Helper class for Groq batch processing operations"""
    
    def __init__(self, api_key):
        self.api_key = api_key
        self.base_url = "https://api.groq.com/openai/v1"
    
    def upload_file(self, file_path):
        """Upload JSONL file to Groq"""
        url = f"{self.base_url}/files"
        
        headers = {"Authorization": f"Bearer {self.api_key}"}
        files = {"file": ("batch_file.jsonl", open(file_path, "rb"))}
        data = {"purpose": "batch"}
        
        response = requests.post(url, headers=headers, files=files, data=data)
        return response.json()
    
    def create_batch(self, input_file_id):
        """Create a batch processing job"""
        url = f"{self.base_url}/batches"
        
        headers = {
            "Authorization": f"Bearer {self.api_key}",
            "Content-Type": "application/json"
        }
        
        data = {
            "input_file_id": input_file_id,
            "endpoint": "/v1/chat/completions",
            "completion_window": "24h"
        }
        
        response = requests.post(url, headers=headers, json=data)
        return response.json()
    
    def check_batch_status(self, batch_id):
        """Check status of a batch job"""
        url = f"{self.base_url}/batches/{batch_id}"
        headers = {"Authorization": f"Bearer {self.api_key}"}
        
        response = requests.get(url, headers=headers)
        return response.json()
    
    def download_results(self, output_file_id):
        """Download batch processing results"""
        url = f"{self.base_url}/files/{output_file_id}/content"
        headers = {"Authorization": f"Bearer {self.api_key}"}
        
        response = requests.get(url, headers=headers)
        
        if response.status_code == 200:
            results = []
            for line in response.text.strip().split('\n'):
                if line.strip():
                    try:
                        results.append(json.loads(line))
                    except json.JSONDecodeError as e:
                        print(f"Error parsing result: {e}")
            return results
        else:
            raise Exception(f"Error downloading results: {response.status_code} - {response.text}")
    
    def process_batch_end_to_end(self, file_path, max_wait_time=7200):
        """Complete batch processing workflow"""
        
        print("🚀 Starting batch processing...")
        
        # Step 1: Upload file
        print("📤 Uploading file...")
        upload_result = self.upload_file(file_path)
        if 'id' not in upload_result:
            raise Exception(f"Upload failed: {upload_result}")
        
        file_id = upload_result['id']
        print(f"✓ File uploaded: {file_id}")
        
        # Step 2: Create batch
        print("⚡ Creating batch job...")
        batch_result = self.create_batch(file_id)
        if 'id' not in batch_result:
            raise Exception(f"Batch creation failed: {batch_result}")
        
        batch_id = batch_result['id']
        print(f"✓ Batch created: {batch_id}")
        
        # Step 3: Monitor progress
        print("⏳ Monitoring batch progress...")
        print(f"   Max wait time: {max_wait_time/60:.1f} minutes")
        start_time = time.time()
        last_status_change = start_time
        status_count = 0
        
        while time.time() - start_time < max_wait_time:
            status = self.check_batch_status(batch_id)
            current_status = status.get('status', 'unknown')
            
            # Get progress info if available
            request_counts = status.get('request_counts', {})
            total = request_counts.get('total', 0)
            completed = request_counts.get('completed', 0)
            failed = request_counts.get('failed', 0)
            
            elapsed_minutes = (time.time() - start_time) / 60
            
            # Show detailed progress every 5 status checks or when status changes
            if status_count % 5 == 0 or current_status != getattr(self, '_last_status', None):
                if total > 0:
                    progress_pct = ((completed + failed) / total) * 100
                    print(f"   Status: {current_status} | Progress: {completed}/{total} ({progress_pct:.1f}%) | Failed: {failed} | Time: {elapsed_minutes:.1f}m")
                else:
                    print(f"   Status: {current_status} | Time: {elapsed_minutes:.1f}m")
                
                self._last_status = current_status
                last_status_change = time.time()
            
            status_count += 1
            
            if current_status == 'completed':
                output_file_id = status.get('output_file_id')
                print(f"✅ Batch completed! Output file: {output_file_id}")
                print(f"   Final stats: {completed} completed, {failed} failed, Total time: {elapsed_minutes:.1f}m")
                
                # --- START: TOKEN USAGE EXTRACTION ---
                usage = status.get('usage', {})
                prompt_tokens = usage.get('prompt_tokens', 0)
                completion_tokens = usage.get('completion_tokens', 0)
                cached_tokens = usage.get('cached_tokens', 0)
                total_tokens = usage.get('total_tokens', 0)
                
                print("\n--- TOKEN USAGE SUMMARY (for all processed requests) ---")
                print(f"   Total Input (Prompt) Tokens: {prompt_tokens:,}")
                print(f"   Total Output (Completion) Tokens: {completion_tokens:,}")
                print(f"   Tokens Served from Cache: {cached_tokens:,} (Savings on Input)")
                
                # Calculate the cache hit rate for the prompt tokens
                if prompt_tokens > 0:
                    cache_hit_rate = (cached_tokens / prompt_tokens) * 100
                    print(f"   Cache Hit Rate (Input): {cache_hit_rate:.1f}%")
                # --- END: TOKEN USAGE EXTRACTION ---
                
                # Step 4: Download results
                print("\n📥 Downloading results...")
                results = self.download_results(output_file_id)
                print(f"✓ Retrieved {len(results)} results")
                
                return {
                    'batch_id': batch_id,
                    'status': status,
                    'results': results
                }
                
            elif current_status == 'failed':
                error_details = status.get('errors', 'No error details')
                raise Exception(f"Batch failed: {error_details}")
            
            elif current_status in ['validating', 'in_progress', 'finalizing']:
                time.sleep(15)  # Check every 15 seconds
            else:
                time.sleep(10)  # Check every 10 seconds for other statuses
        
        # Timeout reached - return batch info for manual checking
        final_status = self.check_batch_status(batch_id)
        elapsed_hours = max_wait_time / 3600
        
        print(f"⏰ Batch processing timed out after {elapsed_hours:.1f} hours")
        print(f"   Batch ID: {batch_id}")
        print(f"   Current status: {final_status.get('status', 'unknown')}")
        print(f"   You can check this batch later using: processor.check_batch_status('{batch_id}')")
        
        raise Exception(f"Batch processing timed out after {elapsed_hours:.1f} hours. Batch ID: {batch_id}")

# Initialize processor and run batch processing
try:
    processor = GroqBatchProcessor(api_key)
    
    # If you have an existing batch ID from a previous run, you can check it instead:
    # existing_batch_id = "batch_01k60hf4fgecgvjev31xqzc0q3"  # Your batch ID from the previous run
    # status = processor.check_batch_status(existing_batch_id)
    # print(f"Existing batch status: {status}")
    
    # Process the batch (this may take 30-60 minutes for large datasets)
    # Increased timeout to 2 hours (7200 seconds)
    batch_results = processor.process_batch_end_to_end(batch_file, max_wait_time=7200)
    
    print("🎉 Batch processing completed successfully!")
    
except Exception as e:
    print(f"❌ Batch processing failed: {e}")
    
    # If it's a timeout error, extract the batch ID for manual checking
    error_msg = str(e)
    if "Batch ID:" in error_msg:
        batch_id = error_msg.split("Batch ID: ")[1].strip()
        print(f"💡 You can check the batch status later using:")
        print(f"   status = processor.check_batch_status('{batch_id}')")
        print(f"   If completed, download with: processor.download_results(status['output_file_id'])")
    
    # Set batch_results to None so we can handle it in the next step
    batch_results = None

🚀 Starting batch processing...
📤 Uploading file...
✓ File uploaded: file_01k6q7w10xeb9rna8fkr49rja9
⚡ Creating batch job...
✓ Batch created: batch_01k6q7w1fmebct0mh43xxxp98x
⏳ Monitoring batch progress...
   Max wait time: 120.0 minutes
   Status: validating | Time: 0.0m
   Status: completed | Progress: 100/100 (100.0%) | Failed: 0 | Time: 0.3m
✅ Batch completed! Output file: file_01k6q7w6m3etj8q09y3aa9h5ef
   Final stats: 100 completed, 0 failed, Total time: 0.3m

--- TOKEN USAGE SUMMARY (for all processed requests) ---
   Total Input (Prompt) Tokens: 0
   Total Output (Completion) Tokens: 0
   Tokens Served from Cache: 0 (Savings on Input)

📥 Downloading results...
✓ Retrieved 100 results
🎉 Batch processing completed successfully!


In [9]:
# ====================================================================
# BATCH ID HELPER - Use this if you have a batch ID from a previous run
# ====================================================================

# If you have a batch ID from a previous run, uncomment and set it here:
# batch_id = "batch_your_id_here"  # Replace with your actual batch ID

# Or check if batch_results exists and extract batch_id from it
if 'batch_results' in locals() and batch_results is not None:
    if 'batch_id' in batch_results:
        batch_id = batch_results['batch_id']
        print(f"✓ Found batch_id in batch_results: {batch_id}")
    else:
        print("⚠️ batch_results exists but no batch_id found in it")
        print("Available keys in batch_results:", list(batch_results.keys()) if isinstance(batch_results, dict) else "Not a dict")
elif 'batch_id' in locals():
    print(f"✓ batch_id already defined: {batch_id}")
else:
    print("ℹ️ No batch_id found. Options:")
    print("   1. Run the batch processing cell (cell 8) to create a new batch")
    print("   2. Manually set batch_id above if you have one from a previous run")
    print("   3. Check if a batch is still running from a previous session")

# Show processor status
if 'processor' in locals():
    print(f"✓ Processor is available")
    if 'batch_id' in locals():
        print(f"💡 Ready to check batch status with: processor.check_batch_status('{batch_id}')")
else:
    print("❌ Processor not available - run the batch processing cell first")

✓ Found batch_id in batch_results: batch_01k6q7w1fmebct0mh43xxxp98x
✓ Processor is available
💡 Ready to check batch status with: processor.check_batch_status('batch_01k6q7w1fmebct0mh43xxxp98x')


In [10]:
# Check if we have batch_results from previous execution, otherwise prompt for batch_id
if 'batch_results' in locals() and batch_results is not None and 'batch_id' in batch_results:
    batch_id = batch_results['batch_id']
    print(f"✓ Using batch_id from batch_results: {batch_id}")
elif 'batch_id' not in locals():
    print("❌ batch_id not found. Please provide one of the following:")
    print("   1. Run the batch processing cell (cell 8) to get a new batch_id")
    print("   2. Or manually set batch_id if you have one from a previous run:")
    print("      batch_id = 'your_batch_id_here'  # Replace with your actual batch ID")
    print("\n💡 If you have a previous batch ID, uncomment and modify the line below:")
    print("# batch_id = 'batch_01k60hf4fgecgvjev31xqzc0q3'  # Example batch ID")
    
    # Let's try to find if processor exists and show recent batches
    if 'processor' in locals():
        print("\n🔍 Processor is available. You can check recent batches or manually set batch_id.")
    else:
        print("\n❌ Processor also not available. Please run the batch processing cell first.")
    
    # Don't continue with the rest of the code if batch_id is missing
    batch_id = None

if batch_id is not None and 'processor' in locals():
    print(f"\n📊 Checking status for batch: {batch_id}")
    status = processor.check_batch_status(batch_id)
    results = processor.download_results(status["output_file_id"])

    prompt_total = 0
    cached_total = 0
    completion_total = 0
    total_total = 0

    for r in results:
        usage_entry = r.get("response", {}).get("body", {}).get("usage", {})
        prompt_total += usage_entry.get("prompt_tokens", 0)
        cached_total += usage_entry.get("cached_tokens", 0)
        completion_total += usage_entry.get("completion_tokens", 0)
        total_total += usage_entry.get("total_tokens", 0)

    print("--- TOTAL USAGE ---")
    print("Prompt tokens:", prompt_total)
    print("Cached tokens:", cached_total)
    print("Completion tokens:", completion_total)
    print("Total tokens:", total_total)
else:
    print("⚠️ Cannot proceed without batch_id and processor. Please:")
    print("   1. Run the batch processing cell to create a new batch, OR")
    print("   2. Set batch_id manually if you have one from a previous run")

✓ Using batch_id from batch_results: batch_01k6q7w1fmebct0mh43xxxp98x

📊 Checking status for batch: batch_01k6q7w1fmebct0mh43xxxp98x
--- TOTAL USAGE ---
Prompt tokens: 39795
Cached tokens: 0
Completion tokens: 31966
Total tokens: 71761


In [11]:
status = processor.check_batch_status(batch_id)
print(json.dumps(status.get("usage", {}), indent=2))

{}


In [12]:
# Ensure processor and batch_id are defined
if 'processor' in locals():
	# Try to get batch_id from batch_results if not defined
	if 'batch_id' not in locals():
		if 'batch_results' in locals() and batch_results is not None and 'batch_id' in batch_results:
			batch_id = batch_results['batch_id']
		else:
			raise NameError("batch_id is not defined and cannot be inferred from batch_results.")
	status = processor.check_batch_status(batch_id)
	usage = status.get("usage", {})

	print("Prompt tokens:", usage.get("prompt_tokens"))
	print("Completion tokens:", usage.get("completion_tokens"))
	print("Cached tokens:", usage.get("cached_tokens"))
else:
	print("❌ 'processor' is not defined. Please run the batch processing cell first.")

Prompt tokens: None
Completion tokens: None
Cached tokens: None


In [13]:
# ====================================================================
# ENHANCED RESULTS PROCESSING - CLASSIFICATION + RATINGS + MISSING INDICATORS
# ====================================================================

def enhanced_merge_batch_results_to_dataframe(df, results, id_mapping):
    """
    Enhanced function to process both topic classification and ratings
    Also creates missing indicators for each topic category
    """
    import json
    
    # Create a copy of the original dataframe
    df_result = df.copy()
    
    # Initialize columns for classification results
    df_result['topic_classification'] = None
    df_result['classification_status'] = 'pending'
    df_result['classification_raw_response'] = None
    
    # Initialize rating columns for each topic
    topic_categories = ['Accessibility', 'Security', 'Convenience', 'Customer Support']
    
    for topic in topic_categories:
        # Rating columns (1-5 scale)
        df_result[f'{topic.lower()}_rating'] = None
        # Missing indicator columns (0 = present, 1 = missing)
        df_result[f'{topic.lower()}_missing'] = 1  # Default to missing
    
    # Process each result
    success_count = 0
    error_count = 0
    json_parse_errors = 0
    
    print("🔄 Processing enhanced batch results...")
    
    for result in results:
        custom_id = result.get('custom_id', '')
        
        if custom_id in id_mapping:
            original_idx = id_mapping[custom_id]
            
            if original_idx in df_result.index:
                response = result.get('response', {})
                
                # Check if response is successful
                if 'body' in response and 'choices' in response['body']:
                    try:
                        raw_content = response['body']['choices'][0]['message']['content'].strip()
                        df_result.loc[original_idx, 'classification_raw_response'] = raw_content
                        
                        # Try to parse JSON response
                        try:
                            parsed_response = json.loads(raw_content)
                            
                            # Extract topics and ratings
                            topics = parsed_response.get('topics', [])
                            ratings = parsed_response.get('ratings', {})
                            
                            # Store topic classification as comma-separated string
                            df_result.loc[original_idx, 'topic_classification'] = ', '.join(topics)
                            
                            # Process each topic category
                            for topic in topic_categories:
                                if topic in topics and topic in ratings:
                                    # Topic is present - store rating and set missing = 0
                                    df_result.loc[original_idx, f'{topic.lower()}_rating'] = ratings[topic]
                                    df_result.loc[original_idx, f'{topic.lower()}_missing'] = 0
                                else:
                                    # Topic is missing - keep rating as None and missing = 1
                                    df_result.loc[original_idx, f'{topic.lower()}_rating'] = None
                                    df_result.loc[original_idx, f'{topic.lower()}_missing'] = 1
                            
                            df_result.loc[original_idx, 'classification_status'] = 'completed'
                            success_count += 1
                            
                        except json.JSONDecodeError as json_e:
                            # Handle non-JSON responses (fallback to old format)
                            df_result.loc[original_idx, 'topic_classification'] = raw_content
                            df_result.loc[original_idx, 'classification_status'] = 'completed_no_json'
                            json_parse_errors += 1
                            
                            # Try to extract topics from comma-separated format
                            fallback_topics = [t.strip() for t in raw_content.split(',') if t.strip()]
                            for topic in topic_categories:
                                if topic in fallback_topics:
                                    df_result.loc[original_idx, f'{topic.lower()}_rating'] = None  # No rating available
                                    df_result.loc[original_idx, f'{topic.lower()}_missing'] = 0
                        
                    except (KeyError, IndexError) as e:
                        df_result.loc[original_idx, 'classification_status'] = f'error_parsing: {str(e)}'
                        error_count += 1
                else:
                    # Handle error responses
                    error_info = response.get('error', {})
                    error_msg = error_info.get('message', 'unknown_error')
                    df_result.loc[original_idx, 'classification_status'] = f'api_error: {error_msg}'
                    error_count += 1
    
    print(f"📊 Enhanced Merge Results:")
    print(f"   ✅ Successfully processed (JSON): {success_count}")
    print(f"   ⚠️ Processed (non-JSON fallback): {json_parse_errors}")
    print(f"   ❌ Errors: {error_count}")
    print(f"   📝 Total processed: {len(results)}")
    
    # Summary statistics
    completed_rows = len(df_result[df_result['classification_status'].str.contains('completed')])
    print(f"\n📈 Results Summary:")
    print(f"   Total classifications: {completed_rows}")
    
    for topic in topic_categories:
        present_count = len(df_result[df_result[f'{topic.lower()}_missing'] == 0])
        missing_count = len(df_result[df_result[f'{topic.lower()}_missing'] == 1])
        avg_rating = df_result[f'{topic.lower()}_rating'].mean()
        
        print(f"   {topic}:")
        print(f"     Present: {present_count} | Missing: {missing_count}")
        if not pd.isna(avg_rating):
            print(f"     Average Rating: {avg_rating:.2f}")
    
    return df_result

print("✅ Enhanced merge function created")
print("🎯 This function will:")
print("   1. Parse JSON responses with topics and ratings")
print("   2. Create rating columns for each topic (1-5 scale)")
print("   3. Create missing indicator columns (0=present, 1=missing)")
print("   4. Handle both JSON and fallback text responses")

✅ Enhanced merge function created
🎯 This function will:
   1. Parse JSON responses with topics and ratings
   2. Create rating columns for each topic (1-5 scale)
   3. Create missing indicator columns (0=present, 1=missing)
   4. Handle both JSON and fallback text responses


In [14]:
# USING ENHANCED MERGE FUNCTION ONLY - OVERWRITES OLD FUNCTIONALITY
# Merge results if batch processing was successful
if batch_results and 'results' in batch_results:
    print("🔗 Merging enhanced results back to dataframe...")
    
    # Get the sample dataframe
    df_sample = df.copy()
    
    # Use ONLY the enhanced merge function for classification + ratings + missing indicators
    df_classified = enhanced_merge_batch_results_to_dataframe(
        df_sample, 
        batch_results['results'], 
        id_mapping
    )
    
    print("✅ Enhanced results merged successfully!")
    
else:
    print("⚠️ No batch results to merge. Using existing data if available...")
    
    # Try to use existing results from previous runs
    if 'df_with_results' in locals():
        df_classified = df_with_results
        print("✓ Using existing classified data")
    else:
        print("❌ No classified data available. Please run batch processing first.")
        df_classified = None

🔗 Merging enhanced results back to dataframe...
🔄 Processing enhanced batch results...
📊 Enhanced Merge Results:
   ✅ Successfully processed (JSON): 87
   ⚠️ Processed (non-JSON fallback): 13
   ❌ Errors: 0
   📝 Total processed: 100

📈 Results Summary:
   Total classifications: 100
   Accessibility:
     Present: 52 | Missing: 48
     Average Rating: 1.33
   Security:
     Present: 40 | Missing: 60
     Average Rating: 1.15
   Convenience:
     Present: 50 | Missing: 50
     Average Rating: 1.70
   Customer Support:
     Present: 18 | Missing: 82
     Average Rating: 1.33
✅ Enhanced results merged successfully!


In [15]:
# Check for exported files and their locations
print("🔍 LOCATING EXPORTED FILES")
print("=" * 50)

# Check current working directory
current_dir = os.getcwd()
print(f"📂 Current working directory: {current_dir}")

# Look for ENHANCED CSV files with classification results + ratings
possible_files = [
    "enhanced_classified_reviews_full.csv",
    "enhanced_classified_reviews_ratings.csv", 
    "enhanced_classified_reviews_results.csv",
    "data/output_data/enhanced_classified_reviews_full.csv",
    "data/output_data/enhanced_classified_reviews_ratings.csv",
    # Legacy files (for backward compatibility)
    "classified_reviews_full.csv",
    "classified_reviews_summary.csv", 
    "data/output_data/classified_reviews_full.csv"
]

print(f"\n📄 Checking for result files:")
found_files = []

for file_path in possible_files:
    full_path = os.path.abspath(file_path)
    if os.path.exists(file_path):
        file_size = os.path.getsize(file_path)
        print(f"   ✅ Found: {full_path} ({file_size:,} bytes)")
        found_files.append(full_path)
    else:
        print(f"   ❌ Not found: {full_path}")

# List all CSV files in current directory and data folder
print(f"\n📁 All CSV files in current directory:")
for file in os.listdir("."):
    if file.endswith(".csv"):
        full_path = os.path.abspath(file)
        file_size = os.path.getsize(file)
        print(f"   • {full_path} ({file_size:,} bytes)")

# Check data folder
data_folders = ["data/output_data", "data/output_proceeds"]
for folder in data_folders:
    if os.path.exists(folder):
        print(f"\n📁 CSV files in {folder}:")
        for file in os.listdir(folder):
            if file.endswith(".csv"):
                file_path = os.path.join(folder, file)
                file_size = os.path.getsize(file_path)
                full_path = os.path.abspath(file_path)
                print(f"   • {full_path} ({file_size:,} bytes)")

if found_files:
    print(f"\n🎯 SUMMARY: Found {len(found_files)} result file(s)")
    print("📋 You can access your results at:")
    for file_path in found_files:
        print(f"   📊 {file_path}")
else:
    print(f"\n⚠️ No result files found. This might mean:")
    print("   1. The batch processing hasn't completed yet")
    print("   2. The analysis function wasn't run successfully")
    print("   3. There was an error during file export")
    
    if 'df_classified' in locals() and df_classified is not None:
        print(f"\n💡 Data is available in memory. Creating export function and running analysis...")
        
        # Create a working export function
        def quick_export_results(df_classified, export_prefix="classified_reviews"):
            if df_classified is None:
                print("❌ No classified data available for analysis")
                return None
            
            print("📈 CLASSIFICATION ANALYSIS")
            print("=" * 50)
            
            # Basic statistics
            total_rows = len(df_classified)
            classified_rows = len(df_classified[df_classified['classification_status'] == 'completed'])
            error_rows = total_rows - classified_rows
            
            print(f"📊 Overall Statistics:")
            print(f"   Total rows: {total_rows}")
            print(f"   Successfully classified: {classified_rows}")
            print(f"   Errors: {error_rows}")
            print(f"   Success rate: {(classified_rows/total_rows)*100:.1f}%")
            
            if classified_rows > 0:
                # Topic analysis
                print(f"\n🏷️ Topic Analysis:")
                
                # Extract and count individual topics
                all_topics = []
                for classification in df_classified['topic_classification'].dropna():
                    if isinstance(classification, str):
                        topics = [topic.strip() for topic in classification.split(',') if topic.strip()]
                        all_topics.extend(topics)
                
                # Topic frequency
                topic_counts = pd.Series(all_topics).value_counts()
                print(f"   Most common topics:")
                for topic, count in topic_counts.head(10).items():
                    print(f"     • {topic}: {count}")
                
                # Sample results
                print(f"\n📝 Sample Classifications:")
                sample_rows = df_classified[df_classified['classification_status'] == 'completed'].head(3)
                for idx, row in sample_rows.iterrows():
                    content_preview = str(row['content'])[:100] + "..."
                    classification = row['topic_classification']
                    print(f"     Row {idx}: {classification}")
                    print(f"          Content: {content_preview}")
            
            # Export files
            print(f"\n💾 Exporting Results:")
            
            # Create output directory if it doesn't exist
            output_dir = "data/output_data"
            os.makedirs(output_dir, exist_ok=True)
            print(f"   ✓ Created directory: {os.path.abspath(output_dir)}")
            
            # Export to current directory (most reliable)
            current_file = f"{output_dir}/{export_prefix}_results.csv"
            current_file_abs = os.path.abspath(current_file)
            df_classified.to_csv(current_file, index=False, encoding='utf-8')
            print(f"   ✅ Exported: {current_file_abs} ({os.path.getsize(current_file):,} bytes)")
            
            # Export to data directory
            data_file = f"{output_dir}/{export_prefix}_full.csv"
            data_file_abs = os.path.abspath(data_file)
            df_classified.to_csv(data_file, index=False, encoding='utf-8')
            print(f"   ✅ Exported: {data_file_abs} ({os.path.getsize(data_file):,} bytes)")
            
            # Export summary
            if 'topic_classification' in df_classified.columns:
                summary_cols = ['content', 'topic_classification', 'classification_status']
                summary_file = f"{output_dir}/{export_prefix}_summary.csv"
                summary_file_abs = os.path.abspath(summary_file)
                df_classified[summary_cols].to_csv(summary_file, index=False, encoding='utf-8')
                print(f"   ✅ Exported: {summary_file_abs} ({os.path.getsize(summary_file):,} bytes)")
            
            print(f"\n🎉 Export completed successfully!")
            return True
        
        # Use ONLY the enhanced export - overwrites old functionality
        try:
            enhanced_analyze_and_export_results(df_classified)
            print("🎯 Enhanced export completed - old functionality overwritten!")
        except Exception as e:
            print(f"❌ Error in enhanced export: {e}")
            # Only minimal fallback if enhanced export completely fails
            try:
                minimal_file = "enhanced_classification_results_minimal.csv"
                df_classified.to_csv(minimal_file, index=False, encoding='utf-8')
                print(f"✅ Minimal enhanced export successful: {os.path.abspath(minimal_file)}")
            except Exception as e2:
                print(f"❌ All export methods failed: {e2}")
    else:
        print("   4. The df_classified variable is not available")

🔍 LOCATING EXPORTED FILES
📂 Current working directory: /Users/mochi/Projects/thesis-bank-sats

📄 Checking for result files:
   ❌ Not found: /Users/mochi/Projects/thesis-bank-sats/enhanced_classified_reviews_full.csv
   ❌ Not found: /Users/mochi/Projects/thesis-bank-sats/enhanced_classified_reviews_ratings.csv
   ✅ Found: /Users/mochi/Projects/thesis-bank-sats/enhanced_classified_reviews_results.csv (129,411 bytes)
   ✅ Found: /Users/mochi/Projects/thesis-bank-sats/data/output_data/enhanced_classified_reviews_full.csv (129,411 bytes)
   ✅ Found: /Users/mochi/Projects/thesis-bank-sats/data/output_data/enhanced_classified_reviews_ratings.csv (26,691 bytes)
   ❌ Not found: /Users/mochi/Projects/thesis-bank-sats/classified_reviews_full.csv
   ❌ Not found: /Users/mochi/Projects/thesis-bank-sats/classified_reviews_summary.csv
   ❌ Not found: /Users/mochi/Projects/thesis-bank-sats/data/output_data/classified_reviews_full.csv

📁 All CSV files in current directory:
   • /Users/mochi/Projects/the

In [16]:
# ====================================================================
# ENHANCED ANALYSIS AND EXPORT - WITH RATINGS AND MISSING INDICATORS
# ====================================================================

def enhanced_analyze_and_export_results(df_classified, export_prefix="enhanced_classified_reviews"):
    """
    Enhanced analysis function for classification + ratings + missing indicators
    """
    
    if df_classified is None:
        print("❌ No classified data available for analysis")
        return None
    
    print("📈 ENHANCED CLASSIFICATION & RATING ANALYSIS")
    print("=" * 60)
    
    # Basic statistics
    total_rows = len(df_classified)
    completed_rows = len(df_classified[df_classified['classification_status'].str.contains('completed', na=False)])
    error_rows = total_rows - completed_rows
    
    print(f"📊 Overall Statistics:")
    print(f"   Total rows: {total_rows}")
    print(f"   Successfully processed: {completed_rows}")
    print(f"   Errors: {error_rows}")
    print(f"   Success rate: {(completed_rows/total_rows)*100:.1f}%")
    
    if completed_rows > 0:
        # Topic presence analysis
        topic_categories = ['Accessibility', 'Security', 'Convenience', 'Customer Support']
        
        print(f"\n🏷️ Topic Presence Analysis:")
        topic_stats = {}
        
        for topic in topic_categories:
            present_count = len(df_classified[df_classified[f'{topic.lower()}_missing'] == 0])
            missing_count = len(df_classified[df_classified[f'{topic.lower()}_missing'] == 1])
            presence_rate = (present_count / total_rows) * 100
            
            topic_stats[topic] = {
                'present': present_count,
                'missing': missing_count,
                'presence_rate': presence_rate
            }
            
            print(f"   {topic}:")
            print(f"     Present: {present_count} ({presence_rate:.1f}%)")
            print(f"     Missing: {missing_count} ({(100-presence_rate):.1f}%)")
        
        # Rating analysis
        print(f"\n⭐ Rating Analysis (1-5 Scale):")
        rating_stats = {}
        
        for topic in topic_categories:
            ratings = df_classified[f'{topic.lower()}_rating'].dropna()
            if len(ratings) > 0:
                avg_rating = ratings.mean()
                rating_distribution = ratings.value_counts().sort_index()
                
                rating_stats[topic] = {
                    'count': len(ratings),
                    'average': avg_rating,
                    'distribution': rating_distribution.to_dict()
                }
                
                print(f"   {topic} (n={len(ratings)}):")
                print(f"     Average Rating: {avg_rating:.2f}")
                print(f"     Distribution: {dict(rating_distribution)}")
            else:
                print(f"   {topic}: No ratings available")
        
        # Sample results
        print(f"\n📝 Sample Enhanced Classifications:")
        sample_rows = df_classified[df_classified['classification_status'].str.contains('completed', na=False)].head(3)
        
        for idx, row in sample_rows.iterrows():
            if 'content' in row:
                content_preview = str(row['content'])[:80] + "..."
            elif 'processed_review' in row:
                content_preview = str(row['processed_review'])[:80] + "..."
            else:
                content_preview = "Content not available"
                
            classification = row['topic_classification']
            print(f"     Row {idx}: {classification}")
            print(f"          Content: {content_preview}")
            
            # Show ratings for this row
            for topic in topic_categories:
                rating = row[f'{topic.lower()}_rating']
                missing = row[f'{topic.lower()}_missing']
                if missing == 0 and pd.notna(rating):
                    print(f"          {topic}: Rating {rating}/5")
    
    # Export files
    print(f"\n💾 Exporting Enhanced Results:")
    
    # Create output directory
    output_dir = "data/output_data"
    os.makedirs(output_dir, exist_ok=True)
    print(f"   ✓ Output directory: {os.path.abspath(output_dir)}")
    
    # Full dataset
    full_file = f"{output_dir}/{export_prefix}_full.csv"
    full_file_abs = os.path.abspath(full_file)
    df_classified.to_csv(full_file, index=False, encoding='utf-8')
    print(f"   ✅ Full dataset: {full_file_abs} ({os.path.getsize(full_file):,} bytes)")
    
    # Ratings summary
    if any(f'{topic.lower()}_rating' in df_classified.columns for topic in ['accessibility', 'security', 'convenience', 'customer support']):
        rating_cols = ['content', 'topic_classification'] if 'content' in df_classified.columns else ['processed_review', 'topic_classification']
        rating_cols.extend([f'{topic.lower()}_rating' for topic in ['accessibility', 'security', 'convenience', 'customer_support']])
        rating_cols.extend([f'{topic.lower()}_missing' for topic in ['accessibility', 'security', 'convenience', 'customer_support']])
        
        # Filter columns that actually exist
        existing_rating_cols = [col for col in rating_cols if col in df_classified.columns]
        
        ratings_file = f"{output_dir}/{export_prefix}_ratings.csv"
        ratings_file_abs = os.path.abspath(ratings_file)
        df_classified[existing_rating_cols].to_csv(ratings_file, index=False, encoding='utf-8')
        print(f"   ✅ Ratings summary: {ratings_file_abs} ({os.path.getsize(ratings_file):,} bytes)")
    
    # Quick access file in current directory
    current_file = f"{export_prefix}_results.csv"
    current_file_abs = os.path.abspath(current_file)
    df_classified.to_csv(current_file, index=False, encoding='utf-8')
    print(f"   ✅ Quick access: {current_file_abs} ({os.path.getsize(current_file):,} bytes)")
    
    # Create analysis summary
    analysis_results = {
        'total_rows': total_rows,
        'completed_rows': completed_rows,
        'error_rows': error_rows,
        'success_rate': (completed_rows/total_rows)*100 if total_rows > 0 else 0,
        'topic_stats': topic_stats if completed_rows > 0 else {},
        'rating_stats': rating_stats if completed_rows > 0 else {}
    }
    
    print(f"\n🎉 Enhanced export completed successfully!")
    return analysis_results

print("✅ Enhanced analysis and export function created")
print("🎯 This function will:")
print("   1. Analyze topic presence rates")
print("   2. Calculate rating statistics for each topic")
print("   3. Export full dataset with all new columns")
print("   4. Create specialized ratings summary file")
print("   5. Provide comprehensive analysis statistics")

✅ Enhanced analysis and export function created
🎯 This function will:
   1. Analyze topic presence rates
   2. Calculate rating statistics for each topic
   3. Export full dataset with all new columns
   4. Create specialized ratings summary file
   5. Provide comprehensive analysis statistics


In [17]:
# ====================================================================
# TEST ENHANCED FUNCTIONALITY (Optional - for testing)
# ====================================================================

def test_enhanced_system_prompt():
    """
    Test the enhanced system prompt with sample data to verify JSON parsing
    """
    print("🧪 TESTING ENHANCED SYSTEM PROMPT")
    print("=" * 50)
    
    # Sample review for testing
    sample_review = "The app is very secure and easy to use, but customer service is terrible"
    
    print(f"📝 Sample review: {sample_review}")
    print(f"\n🤖 Enhanced System Prompt:")
    print(ENHANCED_SYSTEM_PROMPT[:200] + "...")
    
    # Show expected response format
    print(f"\n📋 Expected JSON Response Format:")
    expected_example = {
        "topics": ["Security", "Convenience", "Customer Support"],
        "ratings": {"Security": 4, "Convenience": 4, "Customer Support": 1}
    }
    print(json.dumps(expected_example, indent=2))
    
    # Simulate response processing
    print(f"\n🔄 This would create the following columns:")
    topic_categories = ['Accessibility', 'Security', 'Convenience', 'Customer Support']
    
    print("   Classification columns:")
    print("   - topic_classification: 'Security, Convenience, Customer Support'")
    print("   - classification_status: 'completed'")
    
    print("   Rating columns (1-5):")
    for topic in topic_categories:
        if topic in expected_example['topics']:
            rating = expected_example['ratings'][topic]
            print(f"   - {topic.lower()}_rating: {rating}")
        else:
            print(f"   - {topic.lower()}_rating: None")
    
    print("   Missing indicator columns (0=present, 1=missing):")
    for topic in topic_categories:
        if topic in expected_example['topics']:
            print(f"   - {topic.lower()}_missing: 0")
        else:
            print(f"   - {topic.lower()}_missing: 1")
    
    return True

# Run test if you want to verify the setup
print("💡 Run test_enhanced_system_prompt() to verify the enhanced functionality")
print("🚀 The enhanced workflow is now ready!")
print("\n📋 Summary of enhancements:")
print("   ✅ Enhanced system prompt for classification + rating")
print("   ✅ JSON response parsing for structured data")
print("   ✅ Rating columns for each topic (1-5 scale)")
print("   ✅ Missing indicator columns (0/1 binary)")
print("   ✅ Enhanced analysis and export functions")
print("   ✅ Comprehensive statistics and summaries")

# Uncomment the line below to run the test
# test_enhanced_system_prompt()

💡 Run test_enhanced_system_prompt() to verify the enhanced functionality
🚀 The enhanced workflow is now ready!

📋 Summary of enhancements:
   ✅ Enhanced system prompt for classification + rating
   ✅ JSON response parsing for structured data
   ✅ Rating columns for each topic (1-5 scale)
   ✅ Missing indicator columns (0/1 binary)
   ✅ Enhanced analysis and export functions
   ✅ Comprehensive statistics and summaries


In [18]:
# ====================================================================
# AUTO-EXECUTE ENHANCED EXPORT WHEN DATA IS AVAILABLE
# ====================================================================

# Check if classified data exists and automatically run enhanced export
if 'df_classified' in locals() and df_classified is not None:
    print("🔍 DETECTED CLASSIFIED DATA - AUTO-EXECUTING ENHANCED EXPORT")
    print("=" * 60)
    
    try:
        # Automatically run the enhanced export
        analysis_results = enhanced_analyze_and_export_results(df_classified)
        
        if analysis_results:
            print("🎉 AUTO-EXPORT COMPLETED SUCCESSFULLY!")
            print("📊 Enhanced files have been saved automatically")
        else:
            print("⚠️ Auto-export returned no results")
            
    except Exception as e:
        print(f"❌ Auto-export failed: {e}")
        print("🔄 Attempting manual minimal export...")
        
        try:
            # Manual fallback export
            output_dir = "data/output_data"
            os.makedirs(output_dir, exist_ok=True)
            
            # Save full dataset
            auto_file = f"{output_dir}/auto_enhanced_classified_reviews.csv"
            df_classified.to_csv(auto_file, index=False, encoding='utf-8')
            
            print(f"✅ Manual export successful: {os.path.abspath(auto_file)}")
            print(f"   File size: {os.path.getsize(auto_file):,} bytes")
            print(f"   Rows: {len(df_classified):,}")
            print(f"   Columns: {len(df_classified.columns)}")
            
            # Show column list to verify enhanced columns exist
            enhanced_cols = [col for col in df_classified.columns if 'rating' in col or 'missing' in col]
            if enhanced_cols:
                print(f"   Enhanced columns found: {enhanced_cols}")
            else:
                print("   ⚠️ No enhanced rating/missing columns found")
                
        except Exception as e2:
            print(f"❌ Even manual export failed: {e2}")
else:
    print("ℹ️ No classified data available for auto-export")
    print("   Run the batch processing first to generate classified data")

🔍 DETECTED CLASSIFIED DATA - AUTO-EXECUTING ENHANCED EXPORT
📈 ENHANCED CLASSIFICATION & RATING ANALYSIS
📊 Overall Statistics:
   Total rows: 100
   Successfully processed: 100
   Errors: 0
   Success rate: 100.0%

🏷️ Topic Presence Analysis:
   Accessibility:
     Present: 52 (52.0%)
     Missing: 48 (48.0%)
   Security:
     Present: 40 (40.0%)
     Missing: 60 (60.0%)
   Convenience:
     Present: 50 (50.0%)
     Missing: 50 (50.0%)
   Customer Support:
     Present: 18 (18.0%)
     Missing: 82 (82.0%)

⭐ Rating Analysis (1-5 Scale):
   Accessibility (n=52):
     Average Rating: 1.33
     Distribution: {1: np.int64(44), 2: np.int64(5), 5: np.int64(3)}
   Security (n=40):
     Average Rating: 1.15
     Distribution: {1: np.int64(37), 2: np.int64(2), 5: np.int64(1)}
   Convenience (n=50):
     Average Rating: 1.70
     Distribution: {1: np.int64(34), 2: np.int64(7), 3: np.int64(3), 4: np.int64(2), 5: np.int64(4)}
   Customer Support (n=18):
     Average Rating: 1.33
     Distribution: 

In [19]:
# ====================================================================
# MANUAL EXPORT TRIGGER (Run this cell if auto-export didn't work)
# ====================================================================

def manual_export_now():
    """
    Manual function to export classified data immediately
    """
    print("🚀 MANUAL EXPORT TRIGGERED")
    print("=" * 40)
    
    if 'df_classified' not in locals() or df_classified is None:
        print("❌ No df_classified variable found")
        print("💡 Make sure to run the batch processing and merge cells first")
        return False
    
    print(f"✓ Found df_classified with {len(df_classified)} rows and {len(df_classified.columns)} columns")
    
    # Check for enhanced columns
    enhanced_cols = [col for col in df_classified.columns if any(x in col for x in ['rating', 'missing', 'classification'])]
    print(f"✓ Enhanced columns: {enhanced_cols}")
    
    try:
        # Create output directory
        output_dir = "data/output_data"
        os.makedirs(output_dir, exist_ok=True)
        print(f"✓ Created directory: {output_dir}")
        
        # Export enhanced results
        timestamp = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")
        
        # Full dataset
        full_file = f"{output_dir}/manual_enhanced_classified_{timestamp}.csv"
        df_classified.to_csv(full_file, index=False, encoding='utf-8')
        print(f"✅ Full dataset: {os.path.abspath(full_file)}")
        print(f"   Size: {os.path.getsize(full_file):,} bytes")
        
        # Check if enhanced columns exist and create summary
        rating_cols = [col for col in df_classified.columns if 'rating' in col]
        missing_cols = [col for col in df_classified.columns if 'missing' in col]
        
        if rating_cols or missing_cols:
            summary_cols = ['processed_review', 'topic_classification'] + rating_cols + missing_cols
            existing_cols = [col for col in summary_cols if col in df_classified.columns]
            
            summary_file = f"{output_dir}/manual_enhanced_summary_{timestamp}.csv"
            df_classified[existing_cols].to_csv(summary_file, index=False, encoding='utf-8')
            print(f"✅ Enhanced summary: {os.path.abspath(summary_file)}")
            print(f"   Size: {os.path.getsize(summary_file):,} bytes")
        
        # Show basic stats
        completed = len(df_classified[df_classified['classification_status'].str.contains('completed', na=False)])
        print(f"\n📊 Quick Stats:")
        print(f"   Total rows: {len(df_classified)}")
        print(f"   Completed classifications: {completed}")
        print(f"   Success rate: {(completed/len(df_classified)*100):.1f}%")
        
        # Show rating stats if available
        for col in rating_cols:
            ratings = df_classified[col].dropna()
            if len(ratings) > 0:
                print(f"   {col}: {len(ratings)} ratings, avg: {ratings.mean():.2f}")
        
        return True
        
    except Exception as e:
        print(f"❌ Manual export failed: {e}")
        return False

print("💡 Run manual_export_now() to force export classified data")
print("🎯 This will create timestamped files in data/output_data/")

# Uncomment the line below to run manual export immediately
manual_export_now()

💡 Run manual_export_now() to force export classified data
🎯 This will create timestamped files in data/output_data/
🚀 MANUAL EXPORT TRIGGERED
❌ No df_classified variable found
💡 Make sure to run the batch processing and merge cells first


False

In [20]:
end_time = time.time()

In [21]:
print(f"Duration of entire notebook execution: {(end_time - start_time)/60:.1f} minutes")

Duration of entire notebook execution: 0.4 minutes


In [22]:
%pip list

Package                      Version
---------------------------- --------------
absl-py                      2.3.1
accelerate                   1.10.1
annotated-types              0.7.0
anyio                        4.7.0
app-store-scraper            0.3.5
appnope                      0.1.4
argon2-cffi                  21.3.0
argon2-cffi-bindings         21.2.0
arrow                        1.3.0
asttokens                    3.0.0
astunparse                   1.6.3
async-lru                    2.0.4
attrs                        24.3.0
babel                        2.16.0
beautifulsoup4               4.12.3
bleach                       6.2.0
blis                         1.3.0
Bottleneck                   1.4.2
branca                       0.6.0
Brotli                       1.0.9
brotlicffi                   1.0.9.2
cachetools                   5.5.2
calamanCy                    0.2.2
catalogue                    2.0.10
certifi                      2025.6.15
cffi                         1.

In [23]:
%pip freeze

absl-py==2.3.1
accelerate==1.10.1
annotated-types==0.7.0
anyio @ file:///private/var/folders/k1/30mswbxs7r1g6zwn8y4fyt500000gp/T/abs_d9l4uro_qv/croot/anyio_1745334654441/work
app-store-scraper==0.3.5
appnope @ file:///home/conda/feedstock_root/build_artifacts/appnope_1733332318622/work
argon2-cffi @ file:///opt/conda/conda-bld/argon2-cffi_1645000214183/work
argon2-cffi-bindings @ file:///private/var/folders/nz/j6p8yfhx1mv_0grj5xl4650h0000gp/T/abs_2ef471wnyf/croot/argon2-cffi-bindings_1736182451265/work
arrow==1.3.0
asttokens @ file:///home/conda/feedstock_root/build_artifacts/asttokens_1733250440834/work
astunparse==1.6.3
async-lru @ file:///private/var/folders/k1/30mswbxs7r1g6zwn8y4fyt500000gp/T/abs_02efro5ps8/croot/async-lru_1699554529181/work
attrs @ file:///private/var/folders/nz/j6p8yfhx1mv_0grj5xl4650h0000gp/T/abs_93pjmt0git/croot/attrs_1734533120523/work
babel @ file:///private/var/folders/k1/30mswbxs7r1g6zwn8y4fyt500000gp/T/abs_ed2j11k3aq/croot/babel_1737454371799/work
beautifu